# 03f — Exp 6: Confusion-Weighted Oversampling

**Project:** UREP 32-0210-250078 | Crack Classification

**Method:** After each validation epoch, identify samples in confusion hotspots
(shear->multi_crack, shear->debonding). Oversample these specific misclassified
samples 3x in the next epoch's data loader.

**Why:** Cheap form of online hard example mining. Forces the model to spend extra
capacity on ambiguous boundary cases without requiring re-labeling.

**Setup:** Same 3-phase schedule. Confusion-weighted sampling activates in Phase 3
(where all heads are active and misclassification patterns are most meaningful).

**Expected:** +0.5–1.5 macro-F1, mostly on shear recall.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import numpy as np
import torch

import config
from src.device import print_device_summary, get_device, set_seed
from src.evaluation import evaluate_predictions
from src.model_cbam_hierarchical import (
    InceptionV3CBAMHierarchical, freeze_backbone, unfreeze_segment_b, unfreeze_all,
)
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    get_hierarchical_dataloaders,
    evaluate_hierarchical_model,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
)
from src.losses import (
    compute_class_counts,
    ClassBalancedFocalLoss,
    masked_cb_focal,
    identify_misclassified,
    build_confusion_weighted_sampler,
    train_hierarchical_model_v2,
    HierarchicalCrackDataset,
)
from src.augmentation import get_train_transforms, get_val_test_transforms

set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "exp6_conf_oversample")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
STAGE_BATCH = device_config["batch_sizes"]
NUM_WORKERS = device_config["num_workers"]

print(f"\nExperiment 6: Confusion-Weighted Oversampling")
print(f"Device: {device}")

## Create CB focal loss instances (same as Exp 3)

In [ ]:
train_ds_probe = HierarchicalCrackDataset(
    config.SPLIT_DIR, "train",
    transform=get_val_test_transforms(config.IMG_SIZE, "imagenet"),
)
counts = compute_class_counts(train_ds_probe)
del train_ds_probe

BETA = 0.999
GAMMA = 2.0
OVERSAMPLE_FACTOR = 3

cb_s1 = ClassBalancedFocalLoss(counts["stage1"], beta=BETA, gamma=GAMMA).to(device)
cb_s2 = ClassBalancedFocalLoss(counts["stage2"], beta=BETA, gamma=GAMMA).to(device)
cb_s3 = ClassBalancedFocalLoss(counts["stage3"], beta=BETA, gamma=GAMMA).to(device)

loss_fn_per_stage = {
    "stage1": lambda logits, targets, mask: masked_cb_focal(logits, targets, mask, cb_s1),
    "stage2": lambda logits, targets, mask: masked_cb_focal(logits, targets, mask, cb_s2),
    "stage3": lambda logits, targets, mask: masked_cb_focal(logits, targets, mask, cb_s3),
}

print(f"CB focal: beta={BETA}, gamma={GAMMA}")
print(f"Confusion oversampling factor: {OVERSAMPLE_FACTOR}x")

## Build model

In [ ]:
model = InceptionV3CBAMHierarchical().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## Phase 1 — Feature extraction (standard, no confusion sampling yet)

In [ ]:
train_loader, val_loader, _ = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

freeze_backbone(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=config.STAGE1_LR,
)

history1 = train_hierarchical_model_v2(
    model, train_loader, val_loader, optimizer, device,
    epochs=config.STAGE1_EPOCHS, output_dir=OUTPUT_DIR, stage=1,
    loss_weights=(1.0, 0.0, 0.0),
    loss_fn_per_stage=loss_fn_per_stage,
    model_name="exp6_confos",
)

## Phase 2 — Partial fine-tuning (standard)

In [ ]:
train_loader, val_loader, _ = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[2],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage2",
    multi_factor=2,
)

unfreeze_segment_b(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=config.STAGE2_LR,
)

history2 = train_hierarchical_model_v2(
    model, train_loader, val_loader, optimizer, device,
    epochs=config.STAGE2_EPOCHS, output_dir=OUTPUT_DIR, stage=2,
    loss_weights=(0.3, 0.7, 0.0),
    loss_fn_per_stage=loss_fn_per_stage,
    model_name="exp6_confos",
)

## Phase 3 — Full fine-tuning with confusion-weighted oversampling

After each validation epoch, misclassified training samples get 3x sampling weight.

In [ ]:
# Build the training dataset for Phase 3 (needed for sampler updates)
train_ds_phase3 = HierarchicalCrackDataset(
    config.SPLIT_DIR, "train",
    transform=get_train_transforms(config.IMG_SIZE, "imagenet"),
)

# Initial loader with standard joint sampler
train_loader, val_loader, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[3],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="joint",
    no_crack_mix=0.25,
)

def sampler_update_fn(model, dataset, device):
    """Identify misclassified samples and build boosted sampler."""
    misclassified = identify_misclassified(
        model, dataset, device,
        batch_size=STAGE_BATCH[1],
    )
    return build_confusion_weighted_sampler(
        dataset, misclassified,
        oversample_factor=OVERSAMPLE_FACTOR,
        no_crack_mix=0.25,
    )

unfreeze_all(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.STAGE3_LR)

history3 = train_hierarchical_model_v2(
    model, train_loader, val_loader, optimizer, device,
    epochs=config.STAGE3_EPOCHS, output_dir=OUTPUT_DIR, stage=3,
    loss_weights=(0.2, 0.3, 0.5),
    loss_fn_per_stage=loss_fn_per_stage,
    sampler_update_fn=sampler_update_fn,
    train_dataset=train_ds_phase3,
    batch_size=STAGE_BATCH[3],
    model_name="exp6_confos",
)

## Training curves

In [ ]:
import matplotlib.pyplot as plt

histories = [history1, history2, history3]
phase_names = ["Phase 1 — Feature Extraction", "Phase 2 — Partial FT", "Phase 3 — Full FT + Conf. OS"]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
metric_pairs = [
    ("loss_s1", "val_loss_s1", "Stage 1 loss"),
    ("loss_s2", "val_loss_s2", "Stage 2 loss"),
    ("loss_s3", "val_loss_s3", "Stage 3 loss"),
    ("acc_s1",  "val_acc_s1",  "Stage 1 accuracy"),
    ("acc_s2",  "val_acc_s2",  "Stage 2 accuracy"),
    ("acc_s3",  "val_acc_s3",  "Stage 3 accuracy"),
]

for ax, (tk, vk, title) in zip(axes.flat, metric_pairs):
    offset = 0
    for h, name, color in zip(histories, phase_names, colors):
        n = len(h[tk])
        xs = range(offset, offset + n)
        ax.plot(xs, h[tk], color=color, linestyle="-", label=f"{name} (train)")
        ax.plot(xs, h[vk], color=color, linestyle="--", label=f"{name} (val)")
        if offset > 0:
            ax.axvline(x=offset, color="gray", linestyle=":", alpha=0.5)
        offset += n
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "training_history_exp6.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Save model

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "models", "best_model.pt"))
print("Saved confusion-weighted oversampling model.")

## Test-set evaluation

In [ ]:
_, _, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

results = evaluate_hierarchical_model(model, test_loader, device, t1=0.5, t2=0.5)

y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])
metrics = evaluate_predictions(
    y_true_idx, y_pred_idx,
    output_dir=OUTPUT_DIR, model_name="exp6_conf_oversample",
)

## Per-stage confusion matrices, hierarchical metrics, error attribution

In [ ]:
import seaborn as sns

stage_cms = per_stage_confusion_matrices(results["y_true_paths"], results["y_pred_paths"])
h_metrics = hierarchical_pr_f1(results["y_true_paths"], results["y_pred_paths"])
err_attr  = error_attribution(results["y_true_paths"], results["y_pred_paths"])

print("Hierarchical precision/recall/F1:")
for k, v in h_metrics.items():
    print(f"  {k}: {v:.4f}")

print(f"\nError attribution:")
print(f"  Total:        {err_attr['total']}")
print(f"  Correct:      {err_attr['correct']}")
print(f"  Stage1 errs:  {err_attr['errors_by_stage']['stage1']}")
print(f"  Stage2 errs:  {err_attr['errors_by_stage']['stage2']}")
print(f"  Stage3 errs:  {err_attr['errors_by_stage']['stage3']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["stage1", "stage2", "stage3"]):
    if key not in stage_cms:
        ax.set_visible(False); continue
    info = stage_cms[key]
    sns.heatmap(info["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=info["classes"], yticklabels=info["classes"], ax=ax)
    ax.set_title(f"{key}  ({info['cm'].sum()} samples)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_stage_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

with open(os.path.join(OUTPUT_DIR, "exp6_results.json"), "w") as f:
    json.dump({
        "oversample_factor": OVERSAMPLE_FACTOR,
        "beta": BETA, "gamma": GAMMA,
        "hierarchical": h_metrics,
        "error_attribution": err_attr,
        "flat": {
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        },
    }, f, indent=2)
print(f"\nSaved results to {OUTPUT_DIR}/exp6_results.json")